## Tugas 9 - Predicting Power Consumption using Linear Regression - Supervised Learning

> Download dataset (train_energy_data.csv dan test_energy_data.csv) disini : https://www.kaggle.com/datasets/govindaramsriram/energy-consumption-dataset-linear-regression?select=test_energy_data.csv

> Dataset merupakan data konsumsi daya berdasarkan banyak variabel lainnya.

> Dataset udah di-split menjadi: ``` train_energy_data.csv ``` dan ``` test_energy_data.csv ```

> Isi insight menggunakan bahasa sendiri dan bukan AI - generated.

> Deadline : 31 Oktober 2025.


---

Nama lengkap : Firman Fadilah

Asal universitas : Universitas Udayana

---

### 0. Import Module yang diperlukan

> Import module yang diperlukan untuk mendukung seluruh kegiatan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Problem Identification

> Jelaskan permasalahan yang ingin diselesaikan dari dataset ini. Contoh: "Bagaimana memprediksi konsumsi energi bangunan berdasarkan karakteristik bangunan dan lingkungan sekitar."

Project ini bertujuan untuk membangun model machine learning yang dapat memprediksi konsumsi energi (Energy Consumption) suatu bangunan berdasarkan karakteristik bangunan dan kondisi lingkungannya. Variabel yang digunakan untuk prediksi mencakup luas bangunan (Square Footage), jumlah penghuni (Number of Occupants), jumlah peralatan elektronik yang dipakai (Appliances Used), suhu rata-rata (Average Temperature), tipe bangunan (Building Type), dan hari dalam seminggu (Day of Week).

Algoritma yang dipakai adalah **Linear Regression**, yaitu algoritma supervised learning yang cocok untuk masalah regresi (memprediksi nilai numerik kontinu). Model ini akan belajar dari data training yang sudah punya label Energy Consumption, lalu dievaluasi dengan data testing untuk melihat seberapa akurat prediksinya.

Manfaat dari project ini adalah untuk membantu pengelola bangunan atau penyedia layanan energi mengestimasi konsumsi listrik sebelum operasional, sehingga bisa melakukan perencanaan kapasitas, mengontrol biaya, dan mendukung program efisiensi energi.

### 2. Data Collection

> Di sini silakan muat data dan menuliskan deskripsi awal.

In [ ]:
df_train = pd.read_csv('train_energy_data.csv')
df_test = pd.read_csv('test_energy_data.csv')

print("Shape data training :", df_train.shape)
print("Shape data testing  :", df_test.shape)

df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
kolom_num = df_train.select_dtypes(include='number').columns.tolist()

stats = pd.DataFrame({
    'mean': df_train[kolom_num].mean().round(2),
    'median': df_train[kolom_num].median().round(2),
    'modus': df_train[kolom_num].mode().iloc[0].round(2),
    'variance': df_train[kolom_num].var().round(2),
    'std': df_train[kolom_num].std().round(2),
})
stats

> Dataset training berisi sekitar 800 baris dan testing 200 baris, dengan total 7 kolom. Ada 4 kolom numerik (Square Footage, Number of Occupants, Appliances Used, Average Temperature) dan 2 kolom kategorikal (Building Type, Day of Week). Target yang akan diprediksi adalah Energy Consumption.
>
> Dari statistik deskriptif terlihat bahwa Square Footage punya range yang sangat lebar dari ribuan sampai puluhan ribu square feet, jauh lebih besar skalanya dibanding fitur lain seperti Number of Occupants atau Average Temperature. Karena perbedaan skala ini, nantinya perlu dilakukan normalisasi atau scaling supaya saat training Linear Regression, satu fitur tidak mendominasi yang lain. Tidak ada anomali dari sisi nilai mean, median, dan modus, distribusinya cukup wajar.

### 3. Data Preprocessing

> Tahapan membersihkan data dan menangani data yang hilang atau rusak. Gunakan metode yang diperlukan.

In [ ]:
print("Missing values data training:")
print(df_train.isnull().sum())
print()
print("Missing values data testing:")
print(df_test.isnull().sum())
print()
print("Duplicate training :", df_train.duplicated().sum())
print("Duplicate testing  :", df_test.duplicated().sum())

In [ ]:
kolom_kategori = df_train.select_dtypes(include='object').columns.tolist()
print("Kolom kategorikal:", kolom_kategori)

df_train_enc = pd.get_dummies(df_train, columns=kolom_kategori, drop_first=True)
df_test_enc = pd.get_dummies(df_test, columns=kolom_kategori, drop_first=True)

# Pastikan kolom test sama dengan train (jaga-jaga kalau ada kategori yang tidak muncul)
df_test_enc = df_test_enc.reindex(columns=df_train_enc.columns, fill_value=0)

# Konversi True/False jadi 1/0
df_train_enc = df_train_enc.astype({k: int for k in df_train_enc.select_dtypes(include='bool').columns})
df_test_enc = df_test_enc.astype({k: int for k in df_test_enc.select_dtypes(include='bool').columns})

df_train_enc.head()

In [ ]:
target = 'Energy Consumption'
fitur = [k for k in df_train_enc.columns if k != target]

X_train = df_train_enc[fitur]
y_train = df_train_enc[target]

X_test = df_test_enc[fitur]
y_test = df_test_enc[target]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Shape X_train scaled:", X_train_scaled.shape)
print("Shape X_test scaled :", X_test_scaled.shape)

> Data sudah cukup bersih, tidak ada missing values di kolom manapun dan tidak ada baris yang duplikat, jadi tidak perlu imputasi atau drop data.
>
> Untuk kolom kategorikal Building Type dan Day of Week, dilakukan one-hot encoding pakai pd.get_dummies dengan drop_first=True. Tujuannya supaya tiap kategori punya kolom sendiri yang nilainya 1 atau 0, sekaligus drop kategori pertama supaya tidak ada multicollinearity yang bisa mengganggu performa Linear Regression.
>
> Setelah encoding, dilakukan scaling pakai StandardScaler. Ini penting karena range Square Footage jauh lebih besar dari fitur lain. Tanpa scaling, fitur dengan range besar akan mendominasi koefisien model dan bikin interpretasinya tidak adil. StandardScaler mengubah semua fitur jadi punya mean 0 dan std 1, sehingga semua fitur ada di skala yang setara.

### 4. EDA dan Visualisasi Data

> Menjelajahi data secara mendalam untuk mencari pola dan hubungan antar variabel.

In [ ]:
# Write your code here

In [ ]:
# Write your code here

In [ ]:
# Write your code here

In [ ]:
# Write your code here

> Berikan insight soal kode diatas disini. (klik 2x lalu edit.)

### 5. Pembangunan Model Regresi Linear dan Evaluasi Model

> Latih model menggunakan data yang sudah bersih dan lakukan evaluasi.

In [ ]:
# Write your code here

In [ ]:
# Write your code here

In [ ]:
# Write your code here

> Berikan insight soal kode diatas disini. (klik 2x lalu edit.)

### 6. Simpan model dan scaler

In [ ]:
# Write your code here

### 7. Prediksi Data Baru dan Kesimpulan Akhir

> Coba prediksi data baru yang diinput manual oleh user.

In [ ]:
# Write your code here

### 8. Kesimpulan Project:

> Berikan kesimpulan untuk project ini.

>